In [1]:
# CELL 1 - Imports & Basic Setup

import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import yfinance as yf
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
# CELL 2 - Download & Prepare Yahoo Finance Data

tickers = ["AAPL", "MSFT", "GOOG", "AMZN", "META",
           "TSLA", "NVDA", "JPM", "XOM", "UNH"]

start_date = "2015-01-01"
end_date = "2025-01-01"

raw = yf.download(tickers, start=start_date, end=end_date)

# Fix multi-index columns
raw.columns = ['_'.join(col).strip() for col in raw.columns.values]

# Ensure Adj Close exists (fallback to Close)
for t in tickers:
    if f"Adj Close_{t}" not in raw.columns:
        raw[f"Adj Close_{t}"] = raw[f"Close_{t}"]

# Select OHLCV + Adj Close
features = []
for t in tickers:
    features += [
        f"Open_{t}", f"High_{t}", f"Low_{t}",
        f"Close_{t}", f"Adj Close_{t}", f"Volume_{t}"
    ]

df = raw[features].dropna().astype(np.float32)
print("Final dataframe shape:", df.shape)

/tmp/ipython-input-2052932924.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  10 of 10 completed


Final dataframe shape: (2516, 60)


In [3]:
# CELL 3 - Scaling & Splits

scaler = StandardScaler()
scaled = scaler.fit_transform(df.values)

data = scaled.astype(np.float32)

total_len = len(data)
train_end = int(total_len * 0.8)
val_end = int(total_len * 0.9)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print("Train:", train_data.shape)
print("Val:  ", val_data.shape)
print("Test: ", test_data.shape)

Train: (2012, 60)
Val:   (252, 60)
Test:  (252, 60)


In [4]:
# CELL 4 - Windowed Dataset + DataLoaders

class WindowedDataset(Dataset):
    def __init__(self, array, input_len=96, output_len=24):
        self.array = array
        self.input_len = input_len
        self.output_len = output_len

    def __len__(self):
        return len(self.array) - self.input_len - self.output_len

    def __getitem__(self, idx):
        x = self.array[idx : idx + self.input_len]
        y = self.array[idx + self.input_len : idx + self.input_len + self.output_len]
        return torch.tensor(x), torch.tensor(y)


input_len = 96
output_len = 24
num_assets = 10
features_per_asset = 6

train_ds = WindowedDataset(train_data, input_len, output_len)
val_ds   = WindowedDataset(val_data,   input_len, output_len)
test_ds  = WindowedDataset(test_data,  input_len, output_len)

batch_size = 32

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

print("Dataset sizes:")
print("Train:", len(train_ds))
print("Val:  ", len(val_ds))
print("Test: ", len(test_ds))

Dataset sizes:
Train: 1892
Val:   132
Test:  132


In [5]:
# CELL 5 - DeepLOB-style model for multivariate OHLCV

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dropout=0.1, pool=True):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.pool = nn.MaxPool1d(kernel_size=2) if pool else None

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.act(x)
        x = self.dropout(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act(x)
        x = self.dropout(x)

        if self.pool is not None:
            x = self.pool(x)

        return x


class DeepLOB(nn.Module):
    def __init__(
        self,
        input_len=96,
        output_len=24,
        num_assets=10,
        features_per_asset=6,
        conv_channels=(64, 128, 256),
        lstm_hidden=128,
        lstm_layers=2,
        dropout=0.1,
        bidirectional=True,
    ):
        super().__init__()
        self.input_len = input_len
        self.output_len = output_len
        self.num_assets = num_assets
        self.features_per_asset = features_per_asset

        self.in_features = num_assets * features_per_asset  # 60
        C_in = self.in_features

        blocks = []
        in_ch = C_in
        length = input_len

        for i, out_ch in enumerate(conv_channels):
            pool = (i < len(conv_channels) - 1)
            blocks.append(
                ConvBlock(
                    in_channels=in_ch,
                    out_channels=out_ch,
                    kernel_size=3,
                    dropout=dropout,
                    pool=pool
                )
            )
            in_ch = out_ch
            if pool:
                length = math.floor(length / 2)

        self.conv_net = nn.Sequential(*blocks)
        self.conv_out_channels = in_ch
        self.conv_out_len = length

        self.bidirectional = bidirectional
        self.lstm_hidden = lstm_hidden
        lstm_input_dim = self.conv_out_channels
        lstm_num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=lstm_input_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        fc_in = lstm_hidden * lstm_num_directions
        self.head = nn.Sequential(
            nn.Linear(fc_in, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, output_len * num_assets * features_per_asset),
        )

    def forward(self, x):
        B, T, F = x.shape
        x = x.permute(0, 2, 1)

        x = self.conv_net(x)
        x = x.permute(0, 2, 1)

        lstm_out, _ = self.lstm(x)
        last_step = lstm_out[:, -1, :]

        out = self.head(last_step)
        out = out.reshape(B, self.output_len, self.num_assets * self.features_per_asset)

        return out

In [6]:
# CELL 6 - Instantiate DeepLOB model

model = DeepLOB(
    input_len=input_len,
    output_len=output_len,
    num_assets=num_assets,
    features_per_asset=features_per_asset,
    conv_channels=(64, 128, 256),
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.1,
    bidirectional=True,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

Total parameters: 2055968
Trainable parameters: 2055968


In [7]:
# CELL 7 - Optimizer and Scheduler

criterion_mse = torch.nn.MSELoss()
criterion_mae = torch.nn.L1Loss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def lr_lambda(epoch):
    warmup_epochs = 3
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    return 0.5 * (1 + math.cos(math.pi * (epoch - warmup_epochs) / (num_epochs - warmup_epochs)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [9]:
# CELL 8 - Training and Validation Loop

num_epochs = 50
best_val_mse = float('inf')
best_state = None

for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion_mse(preds, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_losses.append(loss.item())

    scheduler.step()

    model.eval()
    val_losses_mse = []
    val_losses_mae = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            val_losses_mse.append(criterion_mse(preds, yb).item())
            val_losses_mae.append(criterion_mae(preds, yb).item())

    train_mse = sum(train_losses) / len(train_losses)
    val_mse = sum(val_losses_mse) / len(val_losses_mse)
    val_mae = sum(val_losses_mae) / len(val_losses_mae)

    print(f"===== Epoch {epoch+1}/{num_epochs} =====")
    print(f"LR:       {scheduler.get_last_lr()[0]:.6f}")
    print(f"Train MSE:{train_mse:.6f}")
    print(f"Val   MSE:{val_mse:.6f}")
    print(f"Val   MAE:{val_mae:.6f}")

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        best_state = model.state_dict()

===== Epoch 1/50 =====
LR:       0.000667
Train MSE:0.372314
Val   MSE:0.482119
Val   MAE:0.553176
===== Epoch 2/50 =====
LR:       0.001000
Train MSE:0.214242
Val   MSE:0.419308
Val   MAE:0.544336
===== Epoch 3/50 =====
LR:       0.001000
Train MSE:0.183376
Val   MSE:0.498783
Val   MAE:0.583532
===== Epoch 4/50 =====
LR:       0.000999
Train MSE:0.165898
Val   MSE:0.543672
Val   MAE:0.623198
===== Epoch 5/50 =====
LR:       0.000996
Train MSE:0.158382
Val   MSE:0.552939
Val   MAE:0.633387
===== Epoch 6/50 =====
LR:       0.000990
Train MSE:0.171048
Val   MSE:0.539138
Val   MAE:0.617727
===== Epoch 7/50 =====
LR:       0.000982
Train MSE:0.164044
Val   MSE:0.524461
Val   MAE:0.620870
===== Epoch 8/50 =====
LR:       0.000972
Train MSE:0.155134
Val   MSE:0.524174
Val   MAE:0.588833
===== Epoch 9/50 =====
LR:       0.000960
Train MSE:0.158054
Val   MSE:0.546098
Val   MAE:0.612116
===== Epoch 10/50 =====
LR:       0.000946
Train MSE:0.152698
Val   MSE:0.582898
Val   MAE:0.620007
===== Epo

In [10]:
# CELL 9 - Test Evaluation

model.load_state_dict(best_state)
model.eval()

test_mse_list = []
test_mae_list = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb)
        test_mse_list.append(criterion_mse(preds, yb).item())
        test_mae_list.append(criterion_mae(preds, yb).item())

test_mse = sum(test_mse_list) / len(test_mse_list)
test_mae = sum(test_mae_list) / len(test_mae_list)

print("===== DeepLOB Test Performance =====")
print(f"Test MSE: {test_mse:.6f}")
print(f"Test MAE: {test_mae:.6f}")

===== DeepLOB Test Performance =====
Test MSE: 4.124428
Test MAE: 1.816623
